# 6.1 模型版本管理与注册中心

产业级AI部署中，模型版本管理是基础设施而非可选项。一个7B模型从训练到部署涉及数十个版本（不同量化、不同格式、不同硬件适配），没有系统化的版本管理将导致混乱。本章建立模型版本管理的完整方法论。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import hashlib
import json
import os
import time
import tempfile
import shutil
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple
from datetime import datetime
from pathlib import Path

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

## 为什么需要模型版本管理？

| 没有版本管理 | 有版本管理 |
|------------|----------|
| "这个模型是哪个版本的？" → 无人知晓 | 每个模型有唯一版本号和元数据 |
| 线上出问题 → 不知道回滚到哪个版本 | 一键回滚到上一个稳定版本 |
| 新模型上线 → 不确定是否比旧的好 | A/B测试+灰度发布，数据驱动决策 |
| 多平台部署 → 版本不一致 | 统一注册中心，各平台拉取指定版本 |
| 合规审计 → 无法追溯模型变更 | 完整的变更历史和审批记录 |

### 模型版本的复杂性

一个模型在产业级部署中会产生大量衍生版本：

```
Qwen2.5-7B-Instruct (基座模型 v1.0)
├── Q4_K_M.gguf          (llama.cpp CPU版)
├── Q5_K_M.gguf          (llama.cpp CPU版, 高精度)
├── onnx_int8/            (ONNX INT8, 通用中转)
├── qnn_int4.bin          (高通NPU版)
├── coreml_fp16.mlpackage (Apple ANE版)
├── openvino_int8.xml     (Intel NPU版)
├── lora_task_v2.safetensors (LoRA适配器 v2)
└── lora_task_v3.safetensors (LoRA适配器 v3)
```

每个衍生版本都需要独立管理、测试和部署。

## 6.1.1 模型版本命名规范

### 语义化版本号

```
模型版本: {model_name}-{major}.{minor}.{patch}-{quant}-{format}-{target}

示例:
  qwen2.5-7b-1.0.0-fp32-pytorch-origin
  qwen2.5-7b-1.0.0-q4km-gguf-cpu
  qwen2.5-7b-1.0.0-int8-onnx-qnn
  qwen2.5-7b-1.0.0-fp16-coreml-ane
  qwen2.5-7b-1.1.0-q4km-gguf-cpu    (minor: 功能更新)
  qwen2.5-7b-1.0.1-q4km-gguf-cpu    (patch: bug修复)
```

### 版本号规则

| 字段 | 含义 | 变更条件 | 示例 |
|------|------|---------|------|
| **major** | 主版本 | 模型架构变更/基座模型升级 | 1→2 |
| **minor** | 次版本 | 微调更新/量化方案优化 | 0→1 |
| **patch** | 补丁 | bug修复/编译选项调整 | 0→1 |
| **quant** | 量化 | 量化配置 | q4km/int8/fp16 |
| **format** | 格式 | 模型格式 | gguf/onnx/coreml |
| **target** | 目标 | 目标硬件 | cpu/qnn/ane |

In [ ]:
@dataclass
class ModelVersion:
    model_name: str
    major: int
    minor: int
    patch: int
    quant: str
    format: str
    target: str
    base_model_id: Optional[str] = None
    created_at: str = ''
    size_mb: float = 0
    checksum: str = ''
    metrics: Dict = field(default_factory=dict)
    status: str = 'draft'

    @property
    def version_str(self):
        return f"{self.model_name}-{self.major}.{self.minor}.{self.patch}-{self.quant}-{self.format}-{self.target}"

    @property
    def semver(self):
        return f"{self.major}.{self.minor}.{self.patch}"

class ModelRegistry:
    """模型注册中心"""
    def __init__(self):
        self.versions: Dict[str, ModelVersion] = {}
        self.lineage: Dict[str, List[str]] = {}

    def register(self, version: ModelVersion):
        vid = version.version_str
        version.created_at = datetime.now().isoformat()
        self.versions[vid] = version
        if version.base_model_id:
            if version.base_model_id not in self.lineage:
                self.lineage[version.base_model_id] = []
            self.lineage[version.base_model_id].append(vid)

    def promote(self, version_id, stage):
        valid_stages = ['draft', 'staging', 'canary', 'production']
        if version_id in self.versions and stage in valid_stages:
            self.versions[version_id].status = stage

    def get_production_version(self, model_name, quant, format, target):
        for vid, v in self.versions.items():
            if (v.model_name == model_name and v.quant == quant and
                v.format == format and v.target == target and v.status == 'production'):
                return v
        return None

    def rollback(self, model_name, quant, format, target):
        current = self.get_production_version(model_name, quant, format, target)
        if current is None:
            return None
        candidates = [v for v in self.versions.values()
                      if (v.model_name == model_name and v.quant == quant and
                          v.format == format and v.target == target and
                          v.status in ['staging', 'canary'])]
        if not candidates:
            older = [v for v in self.versions.values()
                     if (v.model_name == model_name and v.quant == quant and
                         v.format == format and v.target == target and
                         (v.major, v.minor, v.patch) < (current.major, current.minor, current.patch))]
            candidates = sorted(older, key=lambda v: (v.major, v.minor, v.patch), reverse=True)
        if candidates:
            current.status = 'rolled_back'
            candidates[0].status = 'production'
            return candidates[0]
        return None

registry = ModelRegistry()
base = ModelVersion('qwen2.5-7b', 1, 0, 0, 'fp32', 'pytorch', 'origin',
                     size_mb=28000, metrics={'ppl': 5.12})
registry.register(base)

derivatives = [
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'q4km', 'gguf', 'cpu',
                 base_model_id=base.version_str, size_mb=4200, metrics={'ppl': 5.45, 'tps': 18}),
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'int8', 'onnx', 'qnn',
                 base_model_id=base.version_str, size_mb=7800, metrics={'ppl': 5.38, 'tps': 22}),
    ModelVersion('qwen2.5-7b', 1, 0, 0, 'fp16', 'coreml', 'ane',
                 base_model_id=base.version_str, size_mb=14000, metrics={'ppl': 5.15, 'tps': 15}),
    ModelVersion('qwen2.5-7b', 1, 1, 0, 'q4km', 'gguf', 'cpu',
                 base_model_id=base.version_str, size_mb=4200, metrics={'ppl': 5.30, 'tps': 18}),
]
for d in derivatives:
    registry.register(d)

registry.promote(derivatives[0].version_str, 'production')
registry.promote(derivatives[3].version_str, 'staging')

print("=== 模型注册中心 ===")
print(f"\n{'版本ID':<55} {'状态':<12} {'大小(MB)':<10} {'PPL':<8} {'TPS'}")
print("-" * 95)
for vid, v in registry.versions.items():
    ppl = v.metrics.get('ppl', '-')
    tps = v.metrics.get('tps', '-')
    print(f"{vid:<55} {v.status:<12} {v.size_mb:<10.0f} {ppl:<8} {tps}")

print(f"\n=== 血缘关系 ===")
for parent, children in registry.lineage.items():
    print(f"{parent}")
    for child in children:
        print(f"  └── {child}")

## 6.1.2 模型哈希校验与完整性验证

模型文件在分发和部署过程中可能被篡改或损坏。SHA256哈希校验是确保模型完整性的基础手段，也是模型注册中心的必要组件。

### 为什么需要哈希校验？

1. **防篡改**：攻击者替换模型文件后哈希不匹配
2. **防损坏**：网络传输或存储错误导致文件损坏
3. **防降级**：防止回滚到已知有漏洞的旧版本
4. **审计追踪**：记录每个部署模型的精确哈希

### 哈希算法选择

| 算法 | 速度 | 安全性 | 适用场景 |
|------|------|--------|----------|
| **SHA256** | ~500 MB/s | 高 | 产业标准，HuggingFace使用 |
| **SHA512** | ~400 MB/s | 很高 | 高安全要求 |
| **BLAKE3** | ~1.5 GB/s | 高 | 大文件快速校验 |
| **xxHash** | ~10 GB/s | 低(非密码学) | 快速完整性检查 |

In [ ]:
class ModelIntegrityChecker:
    """模型完整性校验工具"""
    CHUNK_SIZE = 8 * 1024 * 1024

    @staticmethod
    def compute_hash(filepath: str, algorithm: str = 'sha256') -> str:
        h = hashlib.new(algorithm)
        with open(filepath, 'rb') as f:
            while True:
                chunk = f.read(ModelIntegrityChecker.CHUNK_SIZE)
                if not chunk:
                    break
                h.update(chunk)
        return h.hexdigest()

    @staticmethod
    def compute_multi_hash(filepath: str) -> dict:
        hashes = {alg: hashlib.new(alg) for alg in ['sha256', 'sha512', 'md5']}
        with open(filepath, 'rb') as f:
            while True:
                chunk = f.read(ModelIntegrityChecker.CHUNK_SIZE)
                if not chunk:
                    break
                for h in hashes.values():
                    h.update(chunk)
        return {alg: h.hexdigest() for alg, h in hashes.items()}

    @staticmethod
    def verify_model(filepath: str, expected_hash: str, algorithm: str = 'sha256') -> dict:
        t0 = time.perf_counter()
        actual_hash = ModelIntegrityChecker.compute_hash(filepath, algorithm)
        verify_ms = (time.perf_counter() - t0) * 1000
        file_size = os.path.getsize(filepath)
        return {
            'filepath': filepath,
            'algorithm': algorithm,
            'expected': expected_hash,
            'actual': actual_hash,
            'match': actual_hash == expected_hash,
            'file_size_mb': file_size / 1024 / 1024,
            'verify_ms': verify_ms,
            'throughput_mbs': file_size / 1024 / 1024 / (verify_ms / 1000) if verify_ms > 0 else 0,
        }

    @staticmethod
    def create_manifest(model_dir: str) -> dict:
        manifest = {'created_at': datetime.now().isoformat(), 'files': {}}
        for root, dirs, files in os.walk(model_dir):
            for fname in files:
                fpath = os.path.join(root, fname)
                rel_path = os.path.relpath(fpath, model_dir)
                manifest['files'][rel_path] = {
                    'sha256': ModelIntegrityChecker.compute_hash(fpath),
                    'size': os.path.getsize(fpath),
                }
        return manifest

    @staticmethod
    def verify_manifest(model_dir: str, manifest: dict) -> dict:
        results = {'valid': True, 'checked': 0, 'mismatches': [], 'missing': []}
        for rel_path, info in manifest['files'].items():
            fpath = os.path.join(model_dir, rel_path)
            if not os.path.exists(fpath):
                results['missing'].append(rel_path)
                results['valid'] = False
                continue
            actual = ModelIntegrityChecker.compute_hash(fpath)
            if actual != info['sha256']:
                results['mismatches'].append(rel_path)
                results['valid'] = False
            results['checked'] += 1
        return results

print("=== 模型哈希校验实战 ===")

tmpdir = tempfile.mkdtemp()

model_dir = os.path.join(tmpdir, 'qwen2.5-7b-q4km-gguf')
os.makedirs(model_dir, exist_ok=True)

model_files = {
    'model-q4km.gguf': os.urandom(4 * 1024 * 1024),
    'config.json': json.dumps({'model_type': 'qwen2', 'hidden_size': 4096}).encode(),
    'tokenizer.json': json.dumps({'vocab_size': 32000}).encode(),
    'generation_config.json': json.dumps({'max_length': 2048}).encode(),
}
for fname, data in model_files.items():
    with open(os.path.join(model_dir, fname), 'wb') as f:
        f.write(data)

manifest = ModelIntegrityChecker.create_manifest(model_dir)
print(f"模型目录: {model_dir}")
print(f"文件数: {len(manifest['files'])}")
print(f"\n文件清单:")
for rel_path, info in manifest['files'].items():
    print(f"  {rel_path}: {info['size']/1024:.1f} KB, SHA256={info['sha256'][:24]}...")

verify_ok = ModelIntegrityChecker.verify_manifest(model_dir, manifest)
print(f"\n完整性校验: {'✓ 全部通过' if verify_ok['valid'] else '✗ 校验失败'}")
print(f"校验文件数: {verify_ok['checked']}")

corrupt_file = os.path.join(model_dir, 'config.json')
with open(corrupt_file, 'r+b') as f:
    f.write(b'CORRUPTED')
verify_corrupt = ModelIntegrityChecker.verify_manifest(model_dir, manifest)
print(f"\n篡改检测: {'✓ 检测到篡改' if not verify_corrupt['valid'] else '✗ 未检测到'}")
for m in verify_corrupt['mismatches']:
    print(f"  篡改文件: {m}")

print(f"\n--- 多算法哈希性能对比 ---")
test_file = os.path.join(model_dir, 'model-q4km.gguf')
multi = ModelIntegrityChecker.compute_multi_hash(test_file)
for alg, hash_val in multi.items():
    v = ModelIntegrityChecker.verify_model(test_file, hash_val, alg)
    print(f"  {alg}: {v['verify_ms']:.1f}ms, {v['throughput_mbs']:.0f} MB/s, hash={hash_val[:24]}...")

shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\n=== 产业实践要点 ===")
print(f"1. SHA256是产业标准: HuggingFace/MLflow均使用SHA256")
print(f"2. 注册模型时必须计算哈希: register(model) → compute_hash → store checksum")
print(f"3. 部署前必须校验哈希: load_model → verify_hash → match? → deploy/reject")
print(f"4. 清单文件(manifest): 记录目录下所有文件的哈希，支持增量校验")
print(f"5. 大文件校验: 使用流式读取(8MB chunk)，避免一次性加载到内存")
print(f"6. 防篡改链: 训练产出哈希 → 注册中心存储 → 部署时校验 → 运行时抽检")

## 6.1.3 模型部署生命周期

### 四阶段发布流程

```
Draft → Staging → Canary → Production
  │        │         │         │
  │        │         │         └── 100%流量
  │        │         └── 1-5%流量验证
  │        └── 完整测试通过
  └── 初始版本，未测试

回滚路径: Production → Canary → Staging (任一阶段发现问题)
```

### 各阶段的验证要求

| 阶段 | 验证内容 | 通过标准 | 停留时间 |
|------|---------|---------|----------|
| **Draft→Staging** | 精度验证+性能基准 | PPL增加<0.5, 吞吐>基线90% | 自动 |
| **Staging→Canary** | 集成测试+兼容性 | 所有测试用例通过 | 1-2天 |
| **Canary→Production** | 线上小流量验证 | 错误率<0.1%, 延迟P99<SLA | 3-7天 |
| **回滚** | 发现问题 | 任何指标超阈值 | 立即 |

In [ ]:
class DeploymentPipeline:
    """模型部署流水线"""
    STAGES = ['draft', 'staging', 'canary', 'production']

    def __init__(self, registry: ModelRegistry):
        self.registry = registry
        self.validation_results = {}

    def validate_precision(self, version_id, baseline_ppl, threshold=0.5):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'pass': False, 'reason': 'version not found'}
        model_ppl = v.metrics.get('ppl', float('inf'))
        ppl_increase = model_ppl - baseline_ppl
        return {'pass': ppl_increase < threshold, 'ppl_increase': ppl_increase, 'threshold': threshold}

    def validate_performance(self, version_id, baseline_tps, min_ratio=0.9):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'pass': False, 'reason': 'version not found'}
        model_tps = v.metrics.get('tps', 0)
        ratio = model_tps / baseline_tps if baseline_tps > 0 else 0
        return {'pass': ratio >= min_ratio, 'tps_ratio': ratio, 'threshold': min_ratio}

    def promote_with_validation(self, version_id, baseline_ppl, baseline_tps):
        v = self.registry.versions.get(version_id)
        if not v:
            return {'success': False, 'reason': 'version not found'}
        current_stage = v.status
        current_idx = self.STAGES.index(current_stage) if current_stage in self.STAGES else -1
        if current_idx >= len(self.STAGES) - 1:
            return {'success': False, 'reason': 'already in production'}
        precision = self.validate_precision(version_id, baseline_ppl)
        performance = self.validate_performance(version_id, baseline_tps)
        if not precision['pass'] or not performance['pass']:
            return {'success': False, 'precision': precision, 'performance': performance}
        next_stage = self.STAGES[current_idx + 1]
        self.registry.promote(version_id, next_stage)
        return {'success': True, 'from': current_stage, 'to': next_stage,
                'precision': precision, 'performance': performance}

pipeline = DeploymentPipeline(registry)

print("=== 模型部署流水线模拟 ===")
staging_id = [v.version_str for v in registry.versions.values() if v.status == 'staging'][0]
print(f"\n尝试将 {staging_id} 从 staging 提升到 canary...")
result = pipeline.promote_with_validation(staging_id, baseline_ppl=5.45, baseline_tps=18)
if result['success']:
    print(f"  ✓ 提升成功: {result['from']} → {result['to']}")
    print(f"    精度验证: PPL增加={result['precision']['ppl_increase']:.2f} (阈值<{result['precision']['threshold']})")
    print(f"    性能验证: TPS比例={result['performance']['tps_ratio']:.2f} (阈值>{result['performance']['threshold']})")
else:
    print(f"  ✗ 提升失败")
    if 'precision' in result:
        print(f"    精度: PPL增加={result['precision']['ppl_increase']:.2f}")
    if 'performance' in result:
        print(f"    性能: TPS比例={result['performance']['tps_ratio']:.2f}")

print(f"\n=== 模型版本管理总结 ===")
print(f"1. 语义化版本号: model-major.minor.patch-quant-format-target")
print(f"2. 四阶段发布: Draft → Staging → Canary → Production")
print(f"3. 每阶段必须通过精度+性能验证才能提升")
print(f"4. 血缘追踪: 每个衍生版本记录其基座模型")
print(f"5. 一键回滚: 发现问题立即回退到上一稳定版本")
print(f"6. 自动化CI/CD: 模型更新自动触发验证+部署流水线")

## 6.1.4 A/B测试与灰度发布

产业级模型部署中，新模型不能直接全量上线。A/B测试和灰度发布是验证新模型效果的标准方法。

### A/B测试 vs 灰度发布

| 维度 | A/B测试 | 灰度发布 |
|------|---------|----------|
| **目的** | 比较两个模型的效果差异 | 逐步验证新模型稳定性 |
| **流量分配** | 50/50或按比例 | 1%→10%→50%→100% |
| **持续时间** | 固定周期(1-2周) | 每阶段3-7天 |
| **决策依据** | 统计显著性 | 指标不超阈值 |
| **回滚** | 测试结束切换 | 任何阶段可回滚 |

### 灰度发布的关键指标

| 指标 | 计算方式 | 告警阈值 |
|------|---------|----------|
| **PPL漂移** | 新模型PPL - 基线PPL | >0.5 |
| **错误率** | 异常输出/总请求数 | >0.1% |
| **延迟P99** | 99分位响应时间 | >SLA×1.2 |
| **崩溃率** | 崩溃次数/总请求数 | >0.01% |
| **用户负反馈** | 负面评价/总评价 | >5% |

In [ ]:
class CanaryDeployment:
    """灰度发布管理器"""
    TRAFFIC_STAGES = [1, 5, 10, 25, 50, 100]

    def __init__(self, registry: ModelRegistry):
        self.registry = registry
        self.active_deployments: Dict[str, dict] = {}
        self.metrics_history: Dict[str, List[dict]] = {}

    def start_canary(self, version_id: str, baseline_version_id: str) -> dict:
        v = self.registry.versions.get(version_id)
        baseline = self.registry.versions.get(baseline_version_id)
        if not v or not baseline:
            return {'success': False, 'reason': 'version not found'}
        self.active_deployments[version_id] = {
            'baseline_id': baseline_version_id,
            'traffic_pct': 1,
            'stage_idx': 0,
            'start_time': datetime.now().isoformat(),
            'status': 'canary',
        }
        self.metrics_history[version_id] = []
        self.registry.promote(version_id, 'canary')
        return {'success': True, 'traffic_pct': 1, 'stage': '1%'}

    def collect_metrics(self, version_id: str, n_requests: int = 1000) -> dict:
        v = self.registry.versions.get(version_id)
        if not v:
            return None
        base_ppl = v.metrics.get('ppl', 6.0)
        base_tps = v.metrics.get('tps', 18)
        metrics = {
            'timestamp': datetime.now().isoformat(),
            'n_requests': n_requests,
            'avg_ppl': base_ppl + np.random.normal(0, 0.05),
            'p99_latency_ms': 1000 / base_tps * 1000 + np.random.normal(0, 2),
            'error_rate': max(0, np.random.exponential(0.02)),
            'crash_rate': max(0, np.random.exponential(0.005)),
        }
        self.metrics_history[version_id].append(metrics)
        return metrics

    def check_health(self, version_id: str, thresholds: dict = None) -> dict:
        if thresholds is None:
            thresholds = {'max_ppl_drift': 0.5, 'max_error_rate': 0.001,
                          'max_p99_latency_ms': 80, 'max_crash_rate': 0.0001}
        dep = self.active_deployments.get(version_id)
        if not dep or not self.metrics_history.get(version_id):
            return {'healthy': False, 'reason': 'no metrics'}
        latest = self.metrics_history[version_id][-1]
        baseline_v = self.registry.versions.get(dep['baseline_id'])
        baseline_ppl = baseline_v.metrics.get('ppl', 6.0) if baseline_v else 6.0
        checks = {
            'ppl_drift': {'value': latest['avg_ppl'] - baseline_ppl,
                          'threshold': thresholds['max_ppl_drift'],
                          'pass': abs(latest['avg_ppl'] - baseline_ppl) < thresholds['max_ppl_drift']},
            'error_rate': {'value': latest['error_rate'],
                           'threshold': thresholds['max_error_rate'],
                           'pass': latest['error_rate'] < thresholds['max_error_rate']},
            'p99_latency': {'value': latest['p99_latency_ms'],
                            'threshold': thresholds['max_p99_latency_ms'],
                            'pass': latest['p99_latency_ms'] < thresholds['max_p99_latency_ms']},
            'crash_rate': {'value': latest['crash_rate'],
                           'threshold': thresholds['max_crash_rate'],
                           'pass': latest['crash_rate'] < thresholds['max_crash_rate']},
        }
        healthy = all(c['pass'] for c in checks.values())
        return {'healthy': healthy, 'checks': checks}

    def promote_traffic(self, version_id: str) -> dict:
        dep = self.active_deployments.get(version_id)
        if not dep:
            return {'success': False, 'reason': 'not in canary'}
        health = self.check_health(version_id)
        if not health['healthy']:
            return {'success': False, 'reason': 'health check failed', 'health': health}
        dep['stage_idx'] = min(dep['stage_idx'] + 1, len(self.TRAFFIC_STAGES) - 1)
        dep['traffic_pct'] = self.TRAFFIC_STAGES[dep['stage_idx']]
        if dep['traffic_pct'] >= 100:
            dep['status'] = 'production'
            self.registry.promote(version_id, 'production')
        return {'success': True, 'traffic_pct': dep['traffic_pct'], 'status': dep['status']}

    def rollback(self, version_id: str) -> dict:
        dep = self.active_deployments.get(version_id)
        if not dep:
            return {'success': False, 'reason': 'not in canary'}
        dep['status'] = 'rolled_back'
        self.registry.promote(version_id, 'rolled_back')
        baseline = dep['baseline_id']
        self.registry.promote(baseline, 'production')
        return {'success': True, 'rolled_back_to': baseline}

print("=== 灰度发布实战模拟 ===")
canary = CanaryDeployment(registry)

new_version_id = [v.version_str for v in registry.versions.values() if v.status == 'canary']
if not new_version_id:
    new_version_id = derivatives[3].version_str
    prod_version = [v.version_str for v in registry.versions.values() if v.status == 'production'][0]
    result = canary.start_canary(new_version_id, prod_version)
else:
    new_version_id = new_version_id[0]
    prod_version = [v.version_str for v in registry.versions.values() if v.status == 'production'][0]
    result = canary.start_canary(new_version_id, prod_version)

print(f"启动灰度: {result}")

print(f"\n--- 灰度发布流程 ---")
for stage_name, traffic in [('1%流量', 1), ('5%流量', 5), ('10%流量', 10), ('25%流量', 25), ('50%流量', 50), ('100%全量', 100)]:
    metrics = canary.collect_metrics(new_version_id)
    health = canary.check_health(new_version_id)
    status_icon = '✓' if health['healthy'] else '✗'
    print(f"  {stage_name}: PPL漂移={metrics['avg_ppl']-5.45:+.3f}, 错误率={metrics['error_rate']:.4f}, "
          f"P99延迟={metrics['p99_latency_ms']:.1f}ms, 健康={status_icon}")
    if not health['healthy']:
        rb = canary.rollback(new_version_id)
        print(f"  ⚠ 健康检查失败，回滚到 {rb.get('rolled_back_to', 'unknown')}")
        break
    if traffic < 100:
        promo = canary.promote_traffic(new_version_id)
        if not promo['success']:
            print(f"  ⚠ 提升失败: {promo.get('reason', 'unknown')}")
            break

print(f"\n=== A/B测试框架 ===")
class ABTestFramework:
    def __init__(self):
        self.experiments = {}

    def create_experiment(self, name, model_a_id, model_b_id, traffic_split=0.5):
        self.experiments[name] = {
            'model_a': model_a_id, 'model_b': model_b_id,
            'traffic_split': traffic_split,
            'results_a': [], 'results_b': [],
        }

    def route_request(self, name, user_id: int) -> str:
        exp = self.experiments.get(name)
        if not exp:
            return exp['model_a']
        bucket = 'a' if (user_id % 100) < (exp['traffic_split'] * 100) else 'b'
        return exp['model_a'] if bucket == 'a' else exp['model_b']

    def record_result(self, name, model_id, score: float):
        exp = self.experiments.get(name)
        if not exp:
            return
        if model_id == exp['model_a']:
            exp['results_a'].append(score)
        else:
            exp['results_b'].append(score)

    def analyze(self, name) -> dict:
        exp = self.experiments.get(name)
        if not exp or not exp['results_a'] or not exp['results_b']:
            return None
        mean_a = np.mean(exp['results_a'])
        mean_b = np.mean(exp['results_b'])
        std_a = np.std(exp['results_a'])
        std_b = np.std(exp['results_b'])
        n_a, n_b = len(exp['results_a']), len(exp['results_b'])
        se = np.sqrt(std_a**2/n_a + std_b**2/n_b)
        z_score = (mean_b - mean_a) / se if se > 0 else 0
        p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(z_score) / np.sqrt(2))))
        return {
            'mean_a': mean_a, 'mean_b': mean_b,
            'improvement': (mean_b - mean_a) / mean_a * 100,
            'z_score': z_score, 'p_value': p_value,
            'significant': p_value < 0.05,
            'n_a': n_a, 'n_b': n_b,
        }

ab = ABTestFramework()
model_a_id = derivatives[0].version_str
model_b_id = derivatives[3].version_str
ab.create_experiment('ppl_comparison', model_a_id, model_b_id, traffic_split=0.5)

np.random.seed(42)
for user_id in range(2000):
    model_id = ab.route_request('ppl_comparison', user_id)
    if model_id == model_a_id:
        score = 1.0 / (5.45 + np.random.normal(0, 0.1))
    else:
        score = 1.0 / (5.30 + np.random.normal(0, 0.1))
    ab.record_result('ppl_comparison', model_id, score)

analysis = ab.analyze('ppl_comparison')
if analysis:
    print(f"\nA/B测试结果: {model_a_id[:30]}... vs {model_b_id[:30]}...")
    print(f"  模型A均值: {analysis['mean_a']:.4f} (n={analysis['n_a']})")
    print(f"  模型B均值: {analysis['mean_b']:.4f} (n={analysis['n_b']})")
    print(f"  提升: {analysis['improvement']:+.2f}%")
    print(f"  Z-score: {analysis['z_score']:.3f}, P-value: {analysis['p_value']:.4f}")
    print(f"  统计显著: {'✓ 是' if analysis['significant'] else '✗ 否'} (p<0.05)")

print(f"\n=== 产业实践要点 ===")
print(f"1. 灰度发布: 1%→5%→10%→25%→50%→100%，每阶段监控关键指标")
print(f"2. 自动回滚: PPL漂移>0.5 / 错误率>0.1% / P99延迟>SLA×1.2 自动回滚")
print(f"3. A/B测试: 用户级分流(user_id hash)，确保同一用户始终使用同一模型")
print(f"4. 统计显著性: P-value<0.05才认为效果差异显著")
print(f"5. 样本量: 至少1000+请求/组才能得出统计显著结论")
print(f"6. 多指标综合: 不能只看PPL，需综合错误率/延迟/用户反馈")

## 6.1.5 模型制品管理与元数据

一个模型版本不仅仅是权重文件，还包括配置、词表、评估结果、训练日志等制品(Artifact)。系统化的制品管理是可复现部署的基础。

### 模型制品清单

```
qwen2.5-7b-1.0.0-q4km-gguf-cpu/
├── model-q4km.gguf              # 量化权重
├── config.json                  # 模型配置
├── tokenizer.json               # 词表
├── tokenizer_config.json        # 词表配置
├── generation_config.json       # 生成配置
├── special_tokens_map.json      # 特殊token映射
├── model.safetensors.index.json # 分片索引(如适用)
├── metadata.json                # 版本元数据
├── eval_results.json            # 评估结果
├── training_args.json           # 训练参数(如适用)
└── SHA256SUMS                   # 哈希校验文件
```

### 元数据规范

| 字段 | 类型 | 必填 | 说明 |
|------|------|------|------|
| model_name | string | ✓ | 模型名称 |
| version | string | ✓ | 语义化版本号 |
| base_model | string | | 基座模型版本ID |
| quant_config | dict | ✓ | 量化配置 |
| training_config | dict | | 训练配置 |
| eval_metrics | dict | ✓ | 评估指标 |
| created_at | datetime | ✓ | 创建时间 |
| checksum | string | ✓ | SHA256校验和 |
| format | string | ✓ | 模型格式 |
| target_hardware | string | ✓ | 目标硬件 |

In [ ]:
class ModelArtifactManager:
    """模型制品管理器"""
    def __init__(self, base_dir: str):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

    def create_artifact_package(self, version: ModelVersion, model_data: bytes = None) -> dict:
        version_dir = os.path.join(self.base_dir, version.version_str)
        os.makedirs(version_dir, exist_ok=True)

        if model_data is None:
            model_data = os.urandom(int(version.size_mb * 1024 * 100))

        model_path = os.path.join(version_dir, f'model.{version.format}')
        with open(model_path, 'wb') as f:
            f.write(model_data)

        config = {
            'model_type': 'qwen2', 'hidden_size': 4096,
            'intermediate_size': 11008, 'num_attention_heads': 32,
            'num_hidden_layers': 32, 'vocab_size': 32000,
            'quantization_config': {'quant_method': version.quant, 'bits': 4 if 'q4' in version.quant else 8},
        }
        config_path = os.path.join(version_dir, 'config.json')
        with open(config_path, 'w') as f:
            json.dump(config, f, indent=2)

        tokenizer = {'vocab_size': 32000, 'model_max_length': 4096}
        with open(os.path.join(version_dir, 'tokenizer.json'), 'w') as f:
            json.dump(tokenizer, f, indent=2)

        metadata = {
            'model_name': version.model_name,
            'version': version.semver,
            'full_version_id': version.version_str,
            'base_model': version.base_model_id,
            'quant': version.quant,
            'format': version.format,
            'target': version.target,
            'created_at': datetime.now().isoformat(),
            'metrics': version.metrics,
            'status': version.status,
        }
        with open(os.path.join(version_dir, 'metadata.json'), 'w') as f:
            json.dump(metadata, f, indent=2)

        eval_results = {
            'ppl': version.metrics.get('ppl', 0),
            'tps': version.metrics.get('tps', 0),
            'eval_dataset': 'wikitext2',
            'eval_date': datetime.now().isoformat(),
        }
        with open(os.path.join(version_dir, 'eval_results.json'), 'w') as f:
            json.dump(eval_results, f, indent=2)

        checksums = {}
        for fname in os.listdir(version_dir):
            fpath = os.path.join(version_dir, fname)
            if os.path.isfile(fpath):
                h = hashlib.sha256()
                with open(fpath, 'rb') as f:
                    for chunk in iter(lambda: f.read(8*1024*1024), b''):
                        h.update(chunk)
                checksums[fname] = h.hexdigest()
        with open(os.path.join(version_dir, 'SHA256SUMS'), 'w') as f:
            for fname, cksum in checksums.items():
                f.write(f"{cksum}  {fname}\n")

        return {'version_dir': version_dir, 'files': list(checksums.keys()), 'checksums': checksums}

    def verify_package(self, version_id: str) -> dict:
        version_dir = os.path.join(self.base_dir, version_id)
        sums_path = os.path.join(version_dir, 'SHA256SUMS')
        if not os.path.exists(sums_path):
            return {'valid': False, 'reason': 'SHA256SUMS not found'}
        with open(sums_path) as f:
            expected = {}
            for line in f:
                parts = line.strip().split('  ')
                if len(parts) == 2:
                    expected[parts[1]] = parts[0]
        results = {'valid': True, 'checked': 0, 'mismatches': []}
        for fname, expected_hash in expected.items():
            fpath = os.path.join(version_dir, fname)
            if not os.path.exists(fpath):
                results['mismatches'].append(f'{fname}: missing')
                results['valid'] = False
                continue
            h = hashlib.sha256()
            with open(fpath, 'rb') as f:
                for chunk in iter(lambda: f.read(8*1024*1024), b''):
                    h.update(chunk)
            if h.hexdigest() != expected_hash:
                results['mismatches'].append(f'{fname}: hash mismatch')
                results['valid'] = False
            results['checked'] += 1
        return results

print("=== 模型制品管理实战 ===")

tmpdir = tempfile.mkdtemp()
manager = ModelArtifactManager(os.path.join(tmpdir, 'registry'))

test_version = ModelVersion('qwen2.5-7b', 1, 0, 0, 'q4km', 'gguf', 'cpu',
                             size_mb=0.5, metrics={'ppl': 5.45, 'tps': 18})
pkg = manager.create_artifact_package(test_version, model_data=os.urandom(512*1024))

print(f"版本目录: {pkg['version_dir']}")
print(f"\n制品文件:")
for fname in sorted(pkg['files']):
    fpath = os.path.join(pkg['version_dir'], fname)
    size = os.path.getsize(fpath)
    cksum = pkg['checksums'].get(fname, 'N/A')
    print(f"  {fname:<30} {size/1024:>8.1f} KB  SHA256={cksum[:20]}...")

verify = manager.verify_package(test_version.version_str)
print(f"\n完整性校验: {'✓ 全部通过' if verify['valid'] else '✗ 失败'}")
print(f"校验文件数: {verify['checked']}")

metadata_path = os.path.join(pkg['version_dir'], 'metadata.json')
with open(metadata_path) as f:
    metadata = json.load(f)
print(f"\n元数据内容:")
for key, val in metadata.items():
    print(f"  {key}: {val}")

shutil.rmtree(tmpdir, ignore_errors=True)

print(f"\n=== 产业实践要点 ===")
print(f"1. 每个模型版本必须包含完整的制品包: 权重+配置+词表+元数据+校验")
print(f"2. SHA256SUMS文件: 记录目录下所有文件的哈希，支持快速校验")
print(f"3. metadata.json: 包含版本号、基座模型、量化配置、评估指标等")
print(f"4. eval_results.json: 记录评估数据集、指标值、评估日期")
print(f"5. 部署前校验: 读取SHA256SUMS → 逐文件校验 → 全部通过才部署")
print(f"6. HuggingFace规范: 遵循HuggingFace Hub的文件命名和结构规范")

## 6.1.6 模型血缘追踪

模型血缘(Lineage)追踪记录模型从数据→训练→转换→部署的完整链路，是合规审计和问题排查的基础。

### 血缘链路

```
数据集 v2.1 → 训练配置 lr=2e-5, epochs=3 → 基座模型 v1.0
                                              │
                                    ┌─────────┼─────────┐
                                    ▼         ▼         ▼
                              GGUF Q4_K_M  ONNX INT8  CoreML FP16
                              (CPU推理)   (QNN推理)  (ANE推理)
                                    │         │         │
                                    ▼         ▼         ▼
                              LoRA v2    LoRA v2    LoRA v2
                              (任务A)    (任务A)    (任务A)
```

### 血缘元数据

| 字段 | 说明 | 示例 |
|------|------|------|
| dataset_version | 训练数据集版本 | wikitext-v2.1 |
| training_config | 训练超参数 | {lr: 2e-5, epochs: 3} |
| base_model | 基座模型版本ID | qwen2.5-7b-1.0.0-fp32-pytorch-origin |
| conversion_tool | 转换工具及版本 | llama.cpp b2880 |
| conversion_config | 转换配置 | {quant: Q4_K_M, outtype: f16} |

In [ ]:
class ModelLineageTracker:
    """模型血缘追踪器"""
    def __init__(self, registry: ModelRegistry):
        self.registry = registry
        self.lineage_records: Dict[str, dict] = {}

    def record_lineage(self, version_id: str, dataset_version: str = None,
                        training_config: dict = None, conversion_tool: str = None,
                        conversion_config: dict = None):
        v = self.registry.versions.get(version_id)
        if not v:
            return
        self.lineage_records[version_id] = {
            'version_id': version_id,
            'dataset_version': dataset_version,
            'training_config': training_config,
            'base_model': v.base_model_id,
            'conversion_tool': conversion_tool,
            'conversion_config': conversion_config,
            'created_at': v.created_at,
            'metrics': v.metrics,
        }

    def get_full_lineage(self, version_id: str) -> list:
        chain = []
        current = version_id
        visited = set()
        while current and current not in visited:
            visited.add(current)
            record = self.lineage_records.get(current, {})
            v = self.registry.versions.get(current)
            if v:
                chain.append({
                    'version_id': current,
                    'semver': v.semver,
                    'quant': v.quant,
                    'format': v.format,
                    'target': v.target,
                    'base_model': v.base_model_id,
                    'dataset': record.get('dataset_version'),
                    'conversion_tool': record.get('conversion_tool'),
                })
                current = v.base_model_id
            else:
                break
        return chain

    def find_affected_versions(self, base_version_id: str) -> list:
        affected = []
        for vid, children in self.registry.lineage.items():
            if vid == base_version_id or vid.startswith(base_version_id):
                affected.extend(children)
        deeper = []
        for child in affected:
            deeper.extend(self.find_affected_versions(child))
        return list(set(affected + deeper))

    def export_lineage_json(self, version_id: str) -> str:
        chain = self.get_full_lineage(version_id)
        return json.dumps(chain, indent=2, ensure_ascii=False)

tracker = ModelLineageTracker(registry)

tracker.record_lineage(base.version_str,
                       dataset_version='wikitext-v2.1',
                       training_config={'lr': 2e-5, 'epochs': 3, 'batch_size': 128})

for d in derivatives:
    tool_map = {'gguf': 'llama.cpp b2880', 'onnx': 'torch.onnx.export v2.1', 'coreml': 'coremltools 7.2'}
    config_map = {'gguf': {'quant': d.quant, 'outtype': 'f16'},
                  'onnx': {'opset': 17, 'quant': d.quant},
                  'coreml': {'compute_units': 'ALL', 'quant': d.quant}}
    tracker.record_lineage(d.version_str,
                           conversion_tool=tool_map.get(d.format, 'unknown'),
                           conversion_config=config_map.get(d.format, {}))

print("=== 模型血缘追踪 ===")

for d in derivatives[:3]:
    chain = tracker.get_full_lineage(d.version_str)
    print(f"\n--- {d.version_str[:40]}... ---")
    for i, node in enumerate(chain):
        indent = '  ' * i + ('└── ' if i > 0 else '')
        info = f"{node['semver']}-{node['quant']}-{node['format']}-{node['target']}"
        extra = ''
        if node.get('dataset'):
            extra += f" (数据: {node['dataset']})"
        if node.get('conversion_tool'):
            extra += f" [工具: {node['conversion_tool']}]"
        print(f"{indent}{info}{extra}")

print(f"\n--- 影响分析 ---")
affected = tracker.find_affected_versions(base.version_str)
print(f"基座模型 {base.version_str[:40]}... 更新会影响:")
for v in affected:
    print(f"  └── {v[:50]}...")

print(f"\n--- 血缘JSON导出 ---")
lineage_json = tracker.export_lineage_json(derivatives[0].version_str)
print(lineage_json[:300] + '...')

print(f"\n=== 产业实践要点 ===")
print(f"1. 每个模型版本必须记录血缘: 数据集版本+训练配置+基座模型+转换工具")
print(f"2. 影响分析: 基座模型更新时，自动找出所有受影响的衍生版本")
print(f"3. 合规审计: 血缘链路是模型合规审计的核心证据")
print(f"4. 问题排查: 线上模型异常时，沿血缘链路定位问题来源")
print(f"5. 版本锁定: 记录转换工具版本，确保血缘链路可复现")
print(f"6. 自动化: CI/CD管线自动记录血缘，减少人工遗漏")

## 6.1.7 MLflow模型注册中心集成

MLflow是产业级ML生命周期管理平台，其Model Registry组件提供模型版本管理、阶段管理和部署集成。

### MLflow Model Registry核心概念

| 概念 | 说明 | 对应6.1.1概念 |
|------|------|-------------|
| **Registered Model** | 注册的模型(如qwen2.5-7b) | model_name |
| **Model Version** | 模型的特定版本 | ModelVersion |
| **Stage** | 版本阶段 | status (draft/staging/production) |
| **Run** | 训练运行记录 | training_config |
| **Artifact** | 模型制品文件 | 制品包 |

### MLflow vs 自建Registry

| 维度 | MLflow | 自建(S3+DB) |
|------|--------|-------------|
| **上手成本** | 低(开箱即用) | 高(需开发) |
| **灵活性** | 中(标准流程) | 高(完全自定义) |
| **实验追踪** | ✓ 内置 | 需集成 |
| **模型签名** | ✓ 内置 | 需开发 |
| **权限控制** | ✓ 内置 | 需开发 |
| **部署集成** | ✓ 内置 | 需开发 |
| **大规模** | 中(需调优) | 高(可定制) |

In [ ]:
class MLflowSimulator:
    """MLflow Model Registry功能模拟"""
    def __init__(self):
        self.registered_models: Dict[str, dict] = {}
        self.runs: Dict[str, dict] = {}

    def create_registered_model(self, name: str, description: str = ''):
        self.registered_models[name] = {
            'name': name, 'description': description,
            'versions': {}, 'creation_timestamp': datetime.now().isoformat(),
        }

    def log_run(self, run_id: str, params: dict, metrics: dict, tags: dict = None):
        self.runs[run_id] = {
            'run_id': run_id, 'params': params, 'metrics': metrics,
            'tags': tags or {}, 'status': 'FINISHED',
            'start_time': datetime.now().isoformat(),
        }

    def create_model_version(self, name: str, source: str, run_id: str = None,
                              tags: dict = None, description: str = ''):
        if name not in self.registered_models:
            self.create_registered_model(name)
        model = self.registered_models[name]
        version_num = len(model['versions']) + 1
        version_id = str(version_num)
        run_data = self.runs.get(run_id, {})
        model['versions'][version_id] = {
            'version': version_id, 'source': source,
            'run_id': run_id, 'tags': tags or {},
            'description': description, 'status': 'None',
            'creation_timestamp': datetime.now().isoformat(),
            'metrics': run_data.get('metrics', {}),
            'params': run_data.get('params', {}),
        }
        return version_id

    def transition_model_version_stage(self, name: str, version: str, stage: str):
        if name in self.registered_models and version in self.registered_models[name]['versions']:
            old_stage = self.registered_models[name]['versions'][version]['status']
            self.registered_models[name]['versions'][version]['status'] = stage
            return {'name': name, 'version': version, 'from': old_stage, 'to': stage}
        return None

    def get_model_version(self, name: str, version: str):
        if name in self.registered_models and version in self.registered_models[name]['versions']:
            return self.registered_models[name]['versions'][version]
        return None

    def search_model_versions(self, filter_string: str = ''):
        results = []
        for name, model in self.registered_models.items():
            for vid, v in model['versions'].items():
                results.append({'name': name, 'version': vid, 'status': v['status'],
                                'metrics': v.get('metrics', {})})
        return results

mlflow = MLflowSimulator()

mlflow.log_run('run_001', {'lr': 2e-5, 'epochs': 3, 'batch_size': 128},
               {'ppl': 5.12, 'train_loss': 2.8})
mlflow.log_run('run_002', {'lr': 1e-5, 'epochs': 5, 'batch_size': 64},
               {'ppl': 5.30, 'train_loss': 2.6})

mlflow.create_registered_model('qwen2.5-7b', 'Qwen2.5 7B Instruct模型')
v1 = mlflow.create_model_version('qwen2.5-7b', 's3://models/qwen/v1', 'run_001',
                                  tags={'quant': 'fp32', 'format': 'pytorch'})
v2 = mlflow.create_model_version('qwen2.5-7b', 's3://models/qwen/v2', 'run_002',
                                  tags={'quant': 'q4km', 'format': 'gguf'})

mlflow.transition_model_version_stage('qwen2.5-7b', v1, 'Production')
mlflow.transition_model_version_stage('qwen2.5-7b', v2, 'Staging')

print("=== MLflow Model Registry 模拟 ===")
for name, model in mlflow.registered_models.items():
    print(f"\n模型: {name} ({model['description']})")
    for vid, v in model['versions'].items():
        print(f"  版本{vid}: status={v['status']}, run={v['run_id']}, "
              f"tags={v['tags']}, PPL={v['metrics'].get('ppl', 'N/A')}")

print(f"\n--- 搜索模型版本 ---")
versions = mlflow.search_model_versions()
for v in versions:
    print(f"  {v['name']} v{v['version']}: {v['status']}, PPL={v['metrics'].get('ppl', 'N/A')}")

print(f"\n--- 产业级MLflow使用命令 ---")
print(f"# 启动MLflow服务器")
print(f"mlflow server --host 0.0.0.0 --port 5000 --backend-store-uri postgresql://user:pass@db:5432/mlflow")
print(f"")
print(f"# Python SDK使用")
print(f"import mlflow")
print(f"mlflow.set_tracking_uri('http://localhost:5000')")
print(f"")
print(f"# 记录训练运行")
print(f"with mlflow.start_run() as run:")
print(f"    mlflow.log_params({{'lr': 2e-5, 'epochs': 3}})")
print(f"    mlflow.log_metrics({{'ppl': 5.12, 'train_loss': 2.8}})")
print(f"    mlflow.pytorch.log_model(model, 'model')")
print(f"")
print(f"# 注册模型版本")
print(f"mlflow.register_model(f'runs:/{{run.info.run_id}}/model', 'qwen2.5-7b')")
print(f"")
print(f"# 推进阶段")
print(f"client = mlflow.tracking.MlflowClient()")
print(f"client.transition_model_version_stage('qwen2.5-7b', version=2, stage='Production')")

print(f"\n=== 产业实践要点 ===")
print(f"1. MLflow提供完整的实验追踪+模型注册+部署集成")
print(f"2. 自动记录: 训练参数、指标、模型文件、代码版本")
print(f"3. 阶段管理: None → Staging → Production → Archived")
print(f"4. 模型签名: 定义输入/输出schema，部署时自动验证")
print(f"5. 权限控制: 企业版支持RBAC，控制谁可以推进/回滚模型")
print(f"6. 自建替代: S3/MinIO存储 + PostgreSQL元数据 + 自定义API")

## 6.1.8 CI/CD管线集成

模型版本管理必须与CI/CD管线集成，实现从训练到部署的自动化流程。

### CI/CD管线设计

```
代码提交 → 训练触发 → 模型评估 → 格式转换 → 精度验证 → 注册模型 → 灰度部署
   │          │          │          │          │          │          │
   ▼          ▼          ▼          ▼          ▼          ▼          ▼
 git tag   训练集群   PPL<基线?  ONNX/GGUF  cos_sim    注册中心   1%→100%
                                 /TFLite    >0.999     +哈希      监控指标
```

### 关键自动化检查

| 检查项 | 触发条件 | 通过标准 | 失败动作 |
|--------|---------|---------|----------|
| 精度回归 | 每次训练 | PPL增加<0.5 | 阻止注册 |
| 性能回归 | 每次转换 | TPS>基线90% | 阻止部署 |
| 格式兼容 | 每次转换 | ORT推理通过 | 阻止部署 |
| 哈希校验 | 每次注册 | SHA256匹配 | 阻止注册 |
| 安全扫描 | 每次部署 | 无已知漏洞 | 阻止部署 |

In [ ]:
class ModelCICDPipeline:
    """模型CI/CD管线"""
    def __init__(self, registry: ModelRegistry, artifact_manager: ModelArtifactManager = None):
        self.registry = registry
        self.artifact_manager = artifact_manager
        self.pipeline_history: List[dict] = []

    def run_pipeline(self, version: ModelVersion, baseline_ppl: float,
                     baseline_tps: float, skip_conversion: bool = False) -> dict:
        pipeline_id = f"{version.version_str}-{int(time.time())}"
        result = {
            'pipeline_id': pipeline_id,
            'version_id': version.version_str,
            'stages': {},
            'status': 'running',
            'started_at': datetime.now().isoformat(),
        }

        stage1 = self._stage_register(version)
        result['stages']['register'] = stage1
        if not stage1['pass']:
            result['status'] = 'failed'
            self.pipeline_history.append(result)
            return result

        stage2 = self._stage_precision_check(version, baseline_ppl)
        result['stages']['precision_check'] = stage2
        if not stage2['pass']:
            result['status'] = 'failed'
            self.pipeline_history.append(result)
            return result

        stage3 = self._stage_performance_check(version, baseline_tps)
        result['stages']['performance_check'] = stage3
        if not stage3['pass']:
            result['status'] = 'failed'
            self.pipeline_history.append(result)
            return result

        if not skip_conversion and version.format != 'pytorch':
            stage4 = self._stage_conversion_verify(version)
            result['stages']['conversion_verify'] = stage4
            if not stage4['pass']:
                result['status'] = 'failed'
                self.pipeline_history.append(result)
                return result

        stage5 = self._stage_artifact_package(version)
        result['stages']['artifact_package'] = stage5

        stage6 = self._stage_promote(version)
        result['stages']['promote'] = stage6

        result['status'] = 'passed' if stage6['pass'] else 'failed'
        result['completed_at'] = datetime.now().isoformat()
        self.pipeline_history.append(result)
        return result

    def _stage_register(self, version: ModelVersion) -> dict:
        self.registry.register(version)
        return {'pass': True, 'message': f'注册成功: {version.version_str}'}

    def _stage_precision_check(self, version: ModelVersion, baseline_ppl: float) -> dict:
        model_ppl = version.metrics.get('ppl', float('inf'))
        ppl_increase = model_ppl - baseline_ppl
        passed = ppl_increase < 0.5
        return {'pass': passed, 'ppl_increase': ppl_increase,
                'threshold': 0.5, 'message': f'PPL增加={ppl_increase:.3f} (阈值<0.5)'}

    def _stage_performance_check(self, version: ModelVersion, baseline_tps: float) -> dict:
        model_tps = version.metrics.get('tps', 0)
        ratio = model_tps / baseline_tps if baseline_tps > 0 else 0
        passed = ratio >= 0.9
        return {'pass': passed, 'tps_ratio': ratio,
                'threshold': 0.9, 'message': f'TPS比例={ratio:.2f} (阈值>0.9)'}

    def _stage_conversion_verify(self, version: ModelVersion) -> dict:
        return {'pass': True, 'message': f'格式转换验证通过: {version.format}'}

    def _stage_artifact_package(self, version: ModelVersion) -> dict:
        return {'pass': True, 'message': '制品打包完成'}

    def _stage_promote(self, version: ModelVersion) -> dict:
        self.registry.promote(version.version_str, 'staging')
        return {'pass': True, 'message': f'提升到staging: {version.version_str}'}

print("=== CI/CD管线实战模拟 ===")

cicd_registry = ModelRegistry()
pipeline = ModelCICDPipeline(cicd_registry)

test_versions = [
    (ModelVersion('qwen2.5-7b', 2, 0, 0, 'q4km', 'gguf', 'cpu',
                  size_mb=4200, metrics={'ppl': 5.35, 'tps': 18}), 5.45, 18),
    (ModelVersion('qwen2.5-7b', 2, 0, 1, 'q4km', 'gguf', 'cpu',
                  size_mb=4200, metrics={'ppl': 6.50, 'tps': 18}), 5.45, 18),
    (ModelVersion('qwen2.5-7b', 2, 1, 0, 'int8', 'onnx', 'qnn',
                  size_mb=7800, metrics={'ppl': 5.40, 'tps': 12}), 22, 22),
]

for version, base_ppl, base_tps in test_versions:
    result = pipeline.run_pipeline(version, base_ppl, base_tps)
    print(f"\n--- 管线: {version.version_str[:45]}... ---")
    print(f"  状态: {result['status']}")
    for stage_name, stage_result in result['stages'].items():
        icon = '✓' if stage_result['pass'] else '✗'
        print(f"  {icon} {stage_name}: {stage_result['message']}")

print(f"\n=== 管线历史 ===")
for run in pipeline.pipeline_history:
    status_icon = '✓' if run['status'] == 'passed' else '✗'
    n_stages = len(run['stages'])
    n_passed = sum(1 for s in run['stages'].values() if s['pass'])
    print(f"  {status_icon} {run['version_id'][:45]}... {n_passed}/{n_stages} stages passed")

print(f"\n=== 产业级CI/CD配置示例 ===")
print(f"# GitHub Actions")
print(f"name: Model CI/CD")
print(f"on:")
print(f"  push:")
print(f"    paths: ['models/**']")
print(f"jobs:")
print(f"  validate:")
print(f"    runs-on: ubuntu-latest")
print(f"    steps:")
print(f"      - uses: actions/checkout@v4")
print(f"      - name: Precision Check")
print(f"        run: python validate.py --threshold-ppl 0.5")
print(f"      - name: Conversion Verify")
print(f"        run: python convert_and_verify.py --cos-sim-threshold 0.999")
print(f"      - name: Register Model")
print(f"        run: python register_model.py --registry s3://models")
print(f"      - name: Canary Deploy")
print(f"        run: python deploy.py --traffic 1 --auto-rollback")

print(f"\n=== 产业实践要点 ===")
print(f"1. 自动化管线: 训练→评估→转换→验证→注册→部署全流程自动化")
print(f"2. 门控检查: 每个阶段有明确的通过/失败标准，失败立即阻断")
print(f"3. 精度门控: PPL增加<0.5，余弦相似度>0.999")
print(f"4. 性能门控: TPS>基线90%，延迟P99<SLA")
print(f"5. 哈希门控: 注册时计算SHA256，部署时校验SHA256")
print(f"6. 审批流程: Production部署需人工审批，Canary可自动")
print(f"7. 回滚机制: 任何阶段失败自动回滚到上一稳定版本")

---
## 总结

### 模型版本管理核心能力

| 能力 | 工具/方法 | 关键指标 |
|------|----------|----------|
| **版本命名** | 语义化版本号 | model-major.minor.patch-quant-format-target |
| **完整性校验** | SHA256哈希 | 校验时间<1s/GB |
| **阶段管理** | 四阶段发布 | Draft→Staging→Canary→Production |
| **灰度发布** | 流量渐进 | 1%→5%→10%→25%→50%→100% |
| **A/B测试** | 统计显著性 | P-value<0.05 |
| **血缘追踪** | 链路记录 | 数据→训练→转换→部署 |
| **制品管理** | 完整包+SHA256SUMS | 权重+配置+词表+元数据 |
| **CI/CD集成** | 自动化管线 | 门控检查+自动回滚 |

### 产业级最佳实践

1. **版本号是基础**: 语义化版本号+量化+格式+目标硬件，确保唯一标识
2. **哈希校验是必须**: 注册时计算，部署时校验，运行时抽检
3. **灰度发布是标准**: 新模型必须经过小流量验证才能全量上线
4. **血缘追踪是合规要求**: 记录完整的模型→数据→训练→转换链路
5. **CI/CD是效率保障**: 自动化管线减少人为错误，门控检查确保质量
6. **回滚是安全网**: 任何阶段发现问题，秒级回滚到上一稳定版本